In [1]:
import os, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


In [20]:
DATA_PATH = r"C:\Users\fatem\OneDrive\Desktop\Dataset of Arabic Spam and Ham Tweets.csv"

TEXT_COL  = "Tweet Text"    

LABEL_COL = "Label"

MODEL_NAME = "asafaya/bert-base-arabic" 

MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 1
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
CLIP_NORM = 1.0
GRAD_ACCUM_STEPS = 2


In [21]:
df = pd.read_csv(DATA_PATH)
df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)

def map_label(x):
    x = str(x).strip().lower()
    if x == "spam":
        return 1
    if x == "ham":
        return 0
    return None

df["label"] = df[LABEL_COL].apply(map_label)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

df = df.rename(columns={TEXT_COL: "text"})

print("Shape:", df.shape)
print(df["label"].value_counts())


Shape: (13240, 3)
label
0    11299
1     1941
Name: count, dtype: int64


In [22]:
tr_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

print("Train:", tr_df.shape, "Val:", val_df.shape)
print("Train label dist:\n", tr_df["label"].value_counts())
print("Val label dist:\n", val_df["label"].value_counts())


Train: (11916, 3) Val: (1324, 3)
Train label dist:
 label
0    10169
1     1747
Name: count, dtype: int64
Val label dist:
 label
0    1130
1     194
Name: count, dtype: int64


In [23]:
def stratified_cap(df, label_col, cap_per_class, seed=42):
    parts = []
    for lab, grp in df.groupby(label_col):
        if len(grp) > cap_per_class:
            grp = grp.sample(n=cap_per_class, random_state=seed)
        parts.append(grp)
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

CAP_TRAIN_PER_CLASS = 1000
CAP_VAL_PER_CLASS   = 300

tr_small = stratified_cap(tr_df, "label", CAP_TRAIN_PER_CLASS, seed=42)
val_small = stratified_cap(val_df, "label", CAP_VAL_PER_CLASS, seed=42)

print("Train subset:", tr_small.shape)
print("Val subset:", val_small.shape)
print(tr_small["label"].value_counts())


Train subset: (2000, 3)
Val subset: (494, 3)
label
1    1000
0    1000
Name: count, dtype: int64


In [24]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ArabicSpamDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [25]:
train_ds = ArabicSpamDataset(tr_small)
val_ds   = ArabicSpamDataset(val_small)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Batches:", len(train_loader), len(val_loader))


Batches: 250 62


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

y_train = tr_small["label"].values
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

loss_fn = CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.detach().cpu().numpy())


Using device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at asafaya/bert-base-arabic and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: [1. 1.]


In [27]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


In [28]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    loop = tqdm(train_loader, desc=f"Train Epoch {epoch}")
    for step, batch in enumerate(loop, start=1):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = outputs.logits

        loss = loss_fn(logits, labels)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        if step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        loop.set_postfix(loss=f"{(total_loss/step):.4f}")

    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(loader, title="Val"):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=f"Eval {title}"):
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n{title} Accuracy   : {acc:.4f}")
    print(f"{title} Macro F1   : {macro_f1:.4f}")
    print(f"{title} Weighted F1: {weighted_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(all_labels, all_preds, digits=4))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return macro_f1


In [29]:
best_macro = -1.0
save_dir = "arabic_spam_best"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(epoch)
    print(f"\nEpoch {epoch} avg train loss: {avg_loss:.4f}")

    macro = evaluate(val_loader, title="Val")

    if macro > best_macro:
        best_macro = macro
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"\n✅ Saved best model to: {save_dir} (best macro-F1={best_macro:.4f})")


Train Epoch 1: 100%|████████████████████████████████████████████████████| 250/250 [31:30<00:00,  7.56s/it, loss=0.1607]



Epoch 1 avg train loss: 0.1607


Eval Val: 100%|████████████████████████████████████████████████████████████████████████| 62/62 [01:18<00:00,  1.27s/it]



Val Accuracy   : 0.9879
Val Macro F1   : 0.9873
Val Weighted F1: 0.9879

Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.9800    0.9899       300
           1     0.9700    1.0000    0.9848       194

    accuracy                         0.9879       494
   macro avg     0.9850    0.9900    0.9873       494
weighted avg     0.9882    0.9879    0.9879       494

Confusion matrix:
[[294   6]
 [  0 194]]

✅ Saved best model to: arabic_spam_best (best macro-F1=0.9873)


In [32]:
@torch.no_grad()
def predict_text(text: str):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    return torch.argmax(out.logits, dim=1).item()

tests = [
    "مرحبا كيف حالك؟",
    "تهانينا! لقد ربحت جائزة مالية اضغط هنا",
    "تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط",
    "سأتصل بك لاحقاً",
    "اشترك الآن واحصل على خصم 90% لفترة محدودة",
        "مرحبا كيف حالك؟",
    "سأتصل بك لاحقاً",
    "هل وصلت إلى المنزل بسلام؟",
    "لا تنسَ اجتماع الغد الساعة العاشرة",
    "شكراً جزيلاً على مساعدتك اليوم",
    "أنا في الطريق، سأصل بعد عشر دقائق",
    "عيد ميلاد سعيد، أتمنى لك يوماً رائعاً",
    "هل يمكنك إرسال الملف عندما تنتهي؟",
    "سأكون مشغولاً قليلاً، نكمل الحديث لاحقاً",
    "تم استلام رسالتك، شكراً",
        "تهانينا! لقد ربحت جائزة مالية اضغط هنا لاستلامها",
    "تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط",
    "عاجل! فزت بهاتف آيفون جديد، اضغط هنا الآن",
    "تم اختيارك للحصول على قرض فوري بدون ضمانات",
    "لديك مبلغ مسترد بقيمة 500 دولار، أكد بياناتك الآن",
    "تحذير أمني: تم رصد نشاط مشبوه في حسابك",
    "اربح المال من المنزل بسهولة، سجل الآن",
    "تم إيقاف بطاقتك الائتمانية، يرجى التحقق فوراً",
    "اشترك الآن واحصل على خصم 90% لفترة محدودة",
    "رسالة مهمة: تم حظر حسابك مؤقتاً، اضغط لإعادة التفعيل"
]

for t in tests:
    print(t, "=>", predict_text(t), "(0=Ham, 1=Spam)")


مرحبا كيف حالك؟ => 1 (0=Ham, 1=Spam)
تهانينا! لقد ربحت جائزة مالية اضغط هنا => 1 (0=Ham, 1=Spam)
تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط => 1 (0=Ham, 1=Spam)
سأتصل بك لاحقاً => 1 (0=Ham, 1=Spam)
اشترك الآن واحصل على خصم 90% لفترة محدودة => 1 (0=Ham, 1=Spam)
مرحبا كيف حالك؟ => 1 (0=Ham, 1=Spam)
سأتصل بك لاحقاً => 1 (0=Ham, 1=Spam)
هل وصلت إلى المنزل بسلام؟ => 0 (0=Ham, 1=Spam)
لا تنسَ اجتماع الغد الساعة العاشرة => 0 (0=Ham, 1=Spam)
شكراً جزيلاً على مساعدتك اليوم => 1 (0=Ham, 1=Spam)
أنا في الطريق، سأصل بعد عشر دقائق => 1 (0=Ham, 1=Spam)
عيد ميلاد سعيد، أتمنى لك يوماً رائعاً => 1 (0=Ham, 1=Spam)
هل يمكنك إرسال الملف عندما تنتهي؟ => 1 (0=Ham, 1=Spam)
سأكون مشغولاً قليلاً، نكمل الحديث لاحقاً => 0 (0=Ham, 1=Spam)
تم استلام رسالتك، شكراً => 1 (0=Ham, 1=Spam)
تهانينا! لقد ربحت جائزة مالية اضغط هنا لاستلامها => 1 (0=Ham, 1=Spam)
تم تعليق حسابك البنكي، يرجى التحقق فوراً عبر هذا الرابط => 1 (0=Ham, 1=Spam)
عاجل! فزت بهاتف آيفون جديد، اضغط هنا الآن => 1 (0=Ham, 1=Spam)
تم اختيارك 